In [1]:
import pandas as pd
import glob
import os

## Data Loading

In [2]:
all_dfs = []
for filename in glob.glob("data/*.csv"):
    new_df = pd.read_csv(filename)
    all_dfs.append(new_df)
df = pd.concat(all_dfs)

In [3]:
df.to_parquet("data/raw_data.parquet", index=False)

In [6]:
df = pd.read_parquet("data/raw_data.parquet")

In [ ]:
df.head()

In [9]:
df.shape

(9270680, 27)

In [11]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9270680 entries, 0 to 9270679
Data columns (total 27 columns):
 #   Column               Dtype  
---  ------               -----  
 0   FL_DATE              object 
 1   OP_UNIQUE_CARRIER    object 
 2   OP_CARRIER_FL_NUM    int64  
 3   ORIGIN               object 
 4   ORIGIN_CITY_NAME     object 
 5   ORIGIN_STATE_ABR     object 
 6   ORIGIN_STATE_NM      object 
 7   DEST                 object 
 8   DEST_CITY_NAME       object 
 9   DEST_STATE_ABR       object 
 10  DEST_STATE_NM        object 
 11  CRS_DEP_TIME         int64  
 12  DEP_DELAY            float64
 13  DEP_DELAY_NEW        float64
 14  DEP_DEL15            float64
 15  ARR_DELAY            float64
 16  ARR_DEL15            float64
 17  CANCELLED            float64
 18  DIVERTED             float64
 19  CRS_ELAPSED_TIME     float64
 20  DISTANCE             float64
 21  DISTANCE_GROUP       int64  
 22  CARRIER_DELAY        float64
 23  WEATHER_DELAY        float64
 24

In [12]:
print(df.head())

                FL_DATE OP_UNIQUE_CARRIER  OP_CARRIER_FL_NUM ORIGIN  \
0  9/1/2025 12:00:00 AM                AA                  1    JFK   
1  9/1/2025 12:00:00 AM                AA                 10    LAX   
2  9/1/2025 12:00:00 AM                AA               1002    MSN   
3  9/1/2025 12:00:00 AM                AA               1003    CLT   
4  9/1/2025 12:00:00 AM                AA               1003    MCI   

  ORIGIN_CITY_NAME ORIGIN_STATE_ABR ORIGIN_STATE_NM DEST   DEST_CITY_NAME  \
0     New York, NY               NY        New York  LAX  Los Angeles, CA   
1  Los Angeles, CA               CA      California  JFK     New York, NY   
2      Madison, WI               WI       Wisconsin  CLT    Charlotte, NC   
3    Charlotte, NC               NC  North Carolina  MCI  Kansas City, MO   
4  Kansas City, MO               MO        Missouri  CLT    Charlotte, NC   

  DEST_STATE_ABR  ... CANCELLED  DIVERTED  CRS_ELAPSED_TIME  DISTANCE  \
0             CA  ...       0.0      

In [13]:
print(df.describe())

       OP_CARRIER_FL_NUM  CRS_DEP_TIME     DEP_DELAY  DEP_DELAY_NEW  \
count       9.270680e+06  9.270680e+06  9.113111e+06   9.113111e+06   
mean        2.526089e+03  1.323805e+03  1.345565e+01   1.697300e+01   
std         1.630858e+03  4.917860e+02  5.784014e+01   5.665760e+01   
min         1.000000e+00  1.000000e+00 -1.150000e+02   0.000000e+00   
25%         1.207000e+03  9.040000e+02 -6.000000e+00   0.000000e+00   
50%         2.275000e+03  1.317000e+03 -2.000000e+00   0.000000e+00   
75%         3.662000e+03  1.735000e+03  1.000000e+01   1.000000e+01   
max         9.914000e+03  2.400000e+03  4.352000e+03   4.352000e+03   

          DEP_DEL15     ARR_DELAY     ARR_DEL15     CANCELLED      DIVERTED  \
count  9.113111e+06  9.082564e+06  9.082564e+06  9.270680e+06  9.270680e+06   
mean   2.171583e-01  8.041717e+00  2.207319e-01  1.761295e-02  2.678444e-03   
std    4.123113e-01  6.005347e+01  4.147401e-01  1.315399e-01  5.168434e-02   
min    0.000000e+00 -1.280000e+02  0.000000e

In [18]:
missing_values = df.isnull().sum().sort_values(ascending=False)

In [19]:
missing_values

LATE_AIRCRAFT_DELAY    7265868
SECURITY_DELAY         7265868
NAS_DELAY              7265868
WEATHER_DELAY          7265868
CARRIER_DELAY          7265868
ARR_DEL15               188116
ARR_DELAY               188116
DEP_DELAY               157569
DEP_DEL15               157569
DEP_DELAY_NEW           157569
CRS_ELAPSED_TIME             3
DEST_STATE_NM                0
OP_CARRIER_FL_NUM            0
ORIGIN                       0
ORIGIN_CITY_NAME             0
ORIGIN_STATE_ABR             0
DISTANCE_GROUP               0
DISTANCE                     0
DIVERTED                     0
CRS_DEP_TIME                 0
CANCELLED                    0
ORIGIN_STATE_NM              0
DEST                         0
DEST_CITY_NAME               0
OP_UNIQUE_CARRIER            0
DEST_STATE_ABR               0
FL_DATE                      0
dtype: int64

## Data Preprocessing

In [20]:
import numpy as np

In [ ]:
# Group 1 - Delay Cause Columns (Fill with 0 - missing simply means that delay cause didn't apply to that flight)
delay_cause_columns = ['LATE_AIRCRAFT_DELAY', 'SECURITY_DELAY', 'NAS_DELAY', 'WEATHER_DELAY', 'CARRIER_DELAY']
df[delay_cause_columns] = df[delay_cause_columns].fillna(0)

In [ ]:
# Group 2 - Arrival Columns (Filter out cancelled and diverted flights)
df = df[df['CANCELLED'] == 0]
df = df[df['DIVERTED'] == 0]

In [26]:
df.head()

,FL_DATE,OP_UNIQUE_CARRIER,OP_CARRIER_FL_NUM,ORIGIN,ORIGIN_CITY_NAME,ORIGIN_STATE_ABR,ORIGIN_STATE_NM,DEST,DEST_CITY_NAME,DEST_STATE_ABR,...,CANCELLED,DIVERTED,CRS_ELAPSED_TIME,DISTANCE,DISTANCE_GROUP,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY
0,9/1/2025 12:00:00 AM,AA,1,JFK,"New York, NY",NY,New York,LAX,"Los Angeles, CA",CA,...,0.0,0.0,359.0,2475.0,10,0.0,0.0,0.0,0.0,0.0
1,9/1/2025 12:00:00 AM,AA,10,LAX,"Los Angeles, CA",CA,California,JFK,"New York, NY",NY,...,0.0,0.0,334.0,2475.0,10,0.0,0.0,0.0,0.0,0.0
2,9/1/2025 12:00:00 AM,AA,1002,MSN,"Madison, WI",WI,Wisconsin,CLT,"Charlotte, NC",NC,...,0.0,0.0,134.0,708.0,3,0.0,0.0,0.0,0.0,0.0
3,9/1/2025 12:00:00 AM,AA,1003,CLT,"Charlotte, NC",NC,North Carolina,MCI,"Kansas City, MO",MO,...,0.0,0.0,141.0,808.0,4,0.0,0.0,1.0,0.0,34.0
4,9/1/2025 12:00:00 AM,AA,1003,MCI,"Kansas City, MO",MO,Missouri,CLT,"Charlotte, NC",NC,...,0.0,0.0,142.0,808.0,4,0.0,0.0,0.0,0.0,26.0


In [27]:
df.shape

(9082565, 27)

In [ ]:
# Group 3 - Departure Target Columns (Drop these rows -- no target variable means the row is useless for training)
df = df[df['DEP_DELAY'].notna()]
df = df[df['DEP_DEL15'].notna()]
df = df[df['DEP_DELAY_NEW'].notna()]

In [32]:
df = df.dropna(subset=['ARR_DELAY'])
df = df.dropna(subset=['ARR_DEL15'])

In [29]:
# Group 4 - CRS_ELAPSED_TIME (Drop the 3 rows with missing values)
df = df[df['CRS_ELAPSED_TIME'].notna()]

In [30]:
df.shape

(9082564, 27)

In [34]:
df.head()

,FL_DATE,OP_UNIQUE_CARRIER,OP_CARRIER_FL_NUM,ORIGIN,ORIGIN_CITY_NAME,ORIGIN_STATE_ABR,ORIGIN_STATE_NM,DEST,DEST_CITY_NAME,DEST_STATE_ABR,...,CANCELLED,DIVERTED,CRS_ELAPSED_TIME,DISTANCE,DISTANCE_GROUP,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY
0,9/1/2025 12:00:00 AM,AA,1,JFK,"New York, NY",NY,New York,LAX,"Los Angeles, CA",CA,...,0.0,0.0,359.0,2475.0,10,0.0,0.0,0.0,0.0,0.0
1,9/1/2025 12:00:00 AM,AA,10,LAX,"Los Angeles, CA",CA,California,JFK,"New York, NY",NY,...,0.0,0.0,334.0,2475.0,10,0.0,0.0,0.0,0.0,0.0
2,9/1/2025 12:00:00 AM,AA,1002,MSN,"Madison, WI",WI,Wisconsin,CLT,"Charlotte, NC",NC,...,0.0,0.0,134.0,708.0,3,0.0,0.0,0.0,0.0,0.0
3,9/1/2025 12:00:00 AM,AA,1003,CLT,"Charlotte, NC",NC,North Carolina,MCI,"Kansas City, MO",MO,...,0.0,0.0,141.0,808.0,4,0.0,0.0,1.0,0.0,34.0
4,9/1/2025 12:00:00 AM,AA,1003,MCI,"Kansas City, MO",MO,Missouri,CLT,"Charlotte, NC",NC,...,0.0,0.0,142.0,808.0,4,0.0,0.0,0.0,0.0,26.0
